# E34 --- a janela certa quando a lei anda

A porta que o capítulo de ver mundos deixou aberta tem nome: parâmetros que variam no tempo, a
família localmente estacionária. Este caderno atravessa a porta com medição, em três partes:

1. **A varredura.** Nos mundos do livro --- parado, degrau, rampa e a rampa que nunca acaba (a lei
   que anda) --- varrer a janela do estimador de escala e a taxa de esquecimento da variância que
   esquece, medindo o erro contra a verdade declarada e a fração dos dias dentro da tolerância.
   **Critério de separação declarado:** o ótimo de cada forma é a janela de menor erro mediano, com
   faixa de dez a noventa por cento entre mundos; as formas se separam quando as faixas não se
   cruzam. Se o ótimo for o mesmo em todas as formas, a recomendação de janela curta é do mundo ---
   e o capítulo não tem o que medir.

2. **A escada.** A dispersão entre pedaços de dois anos do primeiro mercado (medida no caderno do
   pedaço) contra três contas: dias embaralhados (a conta independente), um AR(1) de coeficiente
   fixo (dependência sem lei que anda) e a família tvAR ajustada (dependência e lei que anda).
   **Critério declarado:** a escada separa quando cada degrau aproxima a dispersão medida mais que
   o degrau anterior.

3. **A região.** Quantos membros da família tvAR (a janela das curvas e a lei do sorteio ---
   gaussiana ou t padronizado --- são o membro) reproduzem as quatro estatísticas da geometria na
   tolerância da casa --- o mesmo método de região do capítulo da tolerância.

**Regras declaradas.** A avaliação de cada configuração começa no primeiro dia em que a sua
estimativa existe depois da mudança --- o aquecimento é custo da janela, não do mundo. O mundo que
anda é a rampa que nunca termina: a escala cresce com passo relativo constante, dobrando do
primeiro ao último dia. A família que se ajusta tem duas curvas --- o coeficiente de média (Yule-Walker rolante) e
a escala de cada dia (o desvio da janela, o instrumento do próprio livro) --- porque nos mercados
é a escala que anda, não a média; a simulação corta o coeficiente no limite declarado e a cauda
entrega um t padronizado quando declarada. Simulação não vira afirmação sobre o mundo (AGENTS.md
§8.5).


In [1]:
# <- brinque com: MUNDOS_VARREDURA, JANELAS, TAXAS, TOLERANCIA, REAMOSTRAS, SEMENTE, SERIES
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, estabilidade, esquecimento, evidencia, graficos, lei, mudanca, promessa, regimes, volatilidade

RAIZ = Path.cwd()
SERIES = ("sp500.csv", "ibov.csv", "btc.csv")
ROTULOS_SERIES = {"sp500.csv": "sp", "ibov.csv": "ibov", "btc.csv": "btc"}
SEMENTE = 97

N_MUNDO = 12000
QUANDO_MUNDO = 3000
FATOR_MUNDO = 2.0
DIAS_RAMPA_CURTA = 252
FORMAS = ("parado", "degrau", "rampa", "andando")
JANELAS = (21, 63, 126, 252, 504, 1008, 2016)
TAXAS = (1.0 / 21, 1.0 / 63, 1.0 / 126, 1.0 / 252, 1.0 / 504, 1.0 / 1008)
MUNDOS_VARREDURA = 40
TOLERANCIA = 0.15
ANOS_VARREDURA = 2

PEDACO_ESCADA = 504
JANELA_ENTREGA = 252
REAMOSTRAS = 24
JANELAS_TVAR = (63, 126, 252, 504)
MUNDOS_REGIAO = 20

SORTEIO = np.random.default_rng(SEMENTE)
print("frevolab %s | %d mundos por forma | janelas %s" % (frevolab.VERSAO, MUNDOS_VARREDURA, JANELAS))

frevolab 0.1.0 | 40 mundos por forma | janelas (21, 63, 126, 252, 504, 1008, 2016)


## Painel 1 --- a janela fixa, forma por forma

O estimador é o desvio da janela; a verdade é a escala da lei de cada dia. No degrau, o erro da
janela de 252 é alto logo depois da mudança e cai com o tempo, porque a janela se renova; no mundo
parado ele fica no piso; no mundo que anda, não há renovação que valha --- a janela atravessa uma
lei diferente da que a calibrou a cada dia que passa. A tabela abre o capítulo com números.

In [2]:
SIGMA = mudanca.SIGMA_PADRAO
VERDADES = {
    "parado": np.full(N_MUNDO, SIGMA),
    "degrau": np.concatenate([np.full(QUANDO_MUNDO, SIGMA),
                              np.full(N_MUNDO - QUANDO_MUNDO, SIGMA * FATOR_MUNDO)]),
    "rampa": np.concatenate([np.full(QUANDO_MUNDO, SIGMA),
                             SIGMA * np.linspace(1.0, FATOR_MUNDO, DIAS_RAMPA_CURTA),
                             np.full(N_MUNDO - QUANDO_MUNDO - DIAS_RAMPA_CURTA, SIGMA * FATOR_MUNDO)]),
    "andando": SIGMA * FATOR_MUNDO ** (np.arange(N_MUNDO) / float(N_MUNDO)),
}

ABERTURA = {}
MUNDOS_ABERTURA = {}
for forma in FORMAS:
    mundos = []
    for _ in range(MUNDOS_VARREDURA):
        if forma == "parado":
            mundos.append(mudanca.estavel(N_MUNDO, SORTEIO))
        elif forma == "degrau":
            mundos.append(mudanca.degrau(N_MUNDO, SORTEIO, fator=FATOR_MUNDO, quando=QUANDO_MUNDO))
        elif forma == "rampa":
            mundos.append(mudanca.rampa(N_MUNDO, SORTEIO, fator=FATOR_MUNDO,
                                        quando=QUANDO_MUNDO, dias=DIAS_RAMPA_CURTA))
        else:
            mundos.append(mudanca.andando(N_MUNDO, SORTEIO, fator=FATOR_MUNDO))
    MUNDOS_ABERTURA[forma] = mundos
    verdade = VERDADES[forma]
    primeiros, ultimos, fracoes = [], [], []
    comeco = QUANDO_MUNDO + 252
    fim_ultimo = N_MUNDO - 2016
    for r in mundos:
        est = volatilidade.volatilidade_rolante(pd.Series(r), 252, dias_uteis=1).to_numpy()
        primeiros.append(esquecimento.erro(est, verdade, inicio=comeco, horizonte=252))
        ultimos.append(esquecimento.erro(est, verdade, inicio=fim_ultimo, horizonte=2016))
        fracoes.append(lei.fracao_na_tolerancia(est, verdade, TOLERANCIA, inicio=comeco,
                                                horizonte=N_MUNDO - comeco))
    ABERTURA[forma] = {"erro_primeiro_ano": float(np.median(primeiros)),
                       "erro_ultimo_trecho": float(np.median(ultimos)),
                       "fracao": float(np.median(fracoes))}
    print("%8s | janela 252: erro %.4f no primeiro ano pos-mudanca, %.4f no ultimo trecho, "
          "fracao na tolerancia %.3f (medianas de %d mundos)"
          % (forma, ABERTURA[forma]["erro_primeiro_ano"], ABERTURA[forma]["erro_ultimo_trecho"],
             ABERTURA[forma]["fracao"], MUNDOS_VARREDURA))

  parado | janela 252: erro 0.0369 no primeiro ano pos-mudanca, 0.0356 no ultimo trecho, fracao na tolerancia 1.000 (medianas de 40 mundos)
  degrau | janela 252: erro 0.0326 no primeiro ano pos-mudanca, 0.0363 no ultimo trecho, fracao na tolerancia 1.000 (medianas de 40 mundos)
   rampa | janela 252: erro 0.0847 no primeiro ano pos-mudanca, 0.0359 no ultimo trecho, fracao na tolerancia 0.994 (medianas de 40 mundos)
 andando | janela 252: erro 0.0302 no primeiro ano pos-mudanca, 0.0352 no ultimo trecho, fracao na tolerancia 1.000 (medianas de 40 mundos)


## Painel 2 --- a varredura: onde o ótimo cai

Janela por janela e taxa por taxa, o erro mediano entre mundos e a fração de dias dentro da
tolerância --- e, de cada mundo, a janela de menor erro, com a sua faixa. O resultado que interessa
é o movimento do ótimo entre formas.

In [3]:
VARREDURA = {}
OTIMOS = {}
for forma in FORMAS:
    mundos = MUNDOS_ABERTURA[forma]
    verdade = VERDADES[forma]
    linhas_j, linhas_t = [], []
    for janela in JANELAS:
        erros, fracoes = [], []
        for r in mundos:
            est = volatilidade.volatilidade_rolante(pd.Series(r), janela, dias_uteis=1).to_numpy()
            comeco = max(QUANDO_MUNDO, janela)
            erros.append(esquecimento.erro(est, verdade, inicio=comeco, horizonte=N_MUNDO - comeco))
            fracoes.append(lei.fracao_na_tolerancia(est, verdade, TOLERANCIA, inicio=comeco,
                                                    horizonte=N_MUNDO - comeco))
        linhas_j.append({"janela": janela, "erro": float(np.median(erros)),
                         "fracao": float(np.median(fracoes)),
                         "erros": erros})
    for taxa in TAXAS:
        erros, fracoes = [], []
        for r in mundos:
            var_esquece = esquecimento.exponencial(r ** 2, taxa)
            est = np.sqrt(var_esquece)
            comeco = QUANDO_MUNDO
            erros.append(esquecimento.erro(est, verdade, inicio=comeco, horizonte=N_MUNDO - comeco))
            fracoes.append(lei.fracao_na_tolerancia(est, verdade, TOLERANCIA, inicio=comeco,
                                                    horizonte=N_MUNDO - comeco))
        linhas_t.append({"taxa": taxa, "erro": float(np.median(erros)),
                         "fracao": float(np.median(fracoes))})
    erros_por_janela = np.array([[l["erros"][m] for l in linhas_j] for m in range(MUNDOS_VARREDURA)])
    otimo_por_mundo = [JANELAS[int(np.argmin(linha))] for linha in erros_por_janela]
    OTIMOS[forma] = {"mediana": float(np.median(otimo_por_mundo)),
                     "piso": float(np.percentile(otimo_por_mundo, 10)),
                     "teto": float(np.percentile(otimo_por_mundo, 90))}
    VARREDURA[forma] = {"janelas": linhas_j, "taxas": linhas_t}
    melhor_taxa = min(linhas_t, key=lambda l: l["erro"])
    print("%8s | otimo da janela: mediana %5.0f dias, faixa [%.0f, %.0f] | "
          "memoria da taxa vencedora: %.0f dias (erro %.4f)"
          % (forma, OTIMOS[forma]["mediana"], OTIMOS[forma]["piso"], OTIMOS[forma]["teto"],
             1.0 / melhor_taxa["taxa"], melhor_taxa["erro"]))

CRITERIO_FORMAS = (OTIMOS["parado"]["piso"] > OTIMOS["degrau"]["teto"]
                   and OTIMOS["degrau"]["piso"] > 0
                   and OTIMOS["andando"]["mediana"] < OTIMOS["parado"]["mediana"])
print("criterio 1 (o otimo se move):", "SEPARA" if CRITERIO_FORMAS else "NAO SEPARA")

  parado | otimo da janela: mediana  2016 dias, faixa [2016, 2016] | memoria da taxa vencedora: 1008 dias (erro 0.0132)


  degrau | otimo da janela: mediana   504 dias, faixa [504, 504] | memoria da taxa vencedora: 252 dias (erro 0.0342)


   rampa | otimo da janela: mediana   504 dias, faixa [504, 504] | memoria da taxa vencedora: 252 dias (erro 0.0363)


 andando | otimo da janela: mediana   504 dias, faixa [504, 1008] | memoria da taxa vencedora: 252 dias (erro 0.0285)
criterio 1 (o otimo se move): SEPARA


## Painel 3 --- a família que anda, ajustada aos mercados

Três contas nos três mercados. Primeiro a curva: o coeficiente rolante e o seu orçamento de
variação --- quanto a lei andou --- com o chão de mundos independentes da mesma janela. Depois a
escada: a dispersão entre pedaços de dois anos da entrega do corte, medida contra os dias
embaralhados, o AR(1) fixo e o tvAR ajustado. Por fim a região: quantos membros tvAR reproduzem as
quatro estatísticas da geometria.

In [4]:
MERCADOS = {}
for arquivo in SERIES:
    retornos = volatilidade.retornos_log(dados.carregar_serie(arquivo)).dropna()
    x = retornos.to_numpy()
    sigma = float(x.std(ddof=1))
    a_global = float(np.corrcoef(x[:-1], x[1:])[0, 1])
    curvas = {j: lei.coeficiente_rolante(x, j) for j in JANELAS_TVAR}
    escalas = {j: volatilidade.volatilidade_rolante(retornos, j, dias_uteis=1).to_numpy() for j in JANELAS_TVAR}
    log_escala = np.log(np.where(np.isfinite(escalas[252]), escalas[252], np.nan))
    log_escalas_chao = []
    for _ in range(REAMOSTRAS):
        calmo = SORTEIO.normal(0.0, 1.0, x.size)
        log_escalas_chao.append(lei.orcamento_variacao(
            np.log(volatilidade.volatilidade_rolante(pd.Series(calmo), 252, dias_uteis=1).to_numpy())))
    MERCADOS[arquivo] = {"dias": int(x.size), "sigma": sigma, "a_global": a_global,
                         "curvas": curvas, "escalas": escalas,
                         "orcamento_escala": lei.orcamento_variacao(log_escala),
                         "orcamento_escala_chao": float(np.median(log_escalas_chao))}
    m = MERCADOS[arquivo]
    print("%-11s %5d dias | a global %.4f | orcamento da lei que anda (log da escala): %.2f contra chao %.2f"
          % (arquivo, m["dias"], a_global, m["orcamento_escala"], m["orcamento_escala_chao"]))

sp500.csv    6718 dias | a global -0.0990 | orcamento da lei que anda (log da escala): 22.59 contra chao 16.44


ibov.csv     6620 dias | a global -0.0235 | orcamento da lei que anda (log da escala): 18.99 contra chao 16.24
btc.csv      4387 dias | a global -0.0222 | orcamento da lei que anda (log da escala): 13.65 contra chao 10.53


In [5]:
ESCADA = {}
for arquivo in MERCADOS:
    x = pd.Series(MERCADOS[arquivo] and volatilidade.retornos_log(dados.carregar_serie(arquivo)).dropna())
    valores = x.to_numpy()
    sigma = MERCADOS[arquivo]["sigma"]
    a_global = MERCADOS[arquivo]["a_global"]
    respostas = estabilidade.por_janela(pd.Series(valores),
                                        lambda p: promessa.entrega(p, JANELA_ENTREGA)["taxa"],
                                        PEDACO_ESCADA)
    degraus = {"medida": float(estabilidade.resumo(respostas, None, None)["dispersao"])}
    ream = {"independente": [], "ar": [], "tvar": []}
    escalas = MERCADOS[arquivo]["escalas"]
    for _ in range(REAMOSTRAS):
        ream["independente"].append(float(estabilidade.resumo(estabilidade.por_janela(
            pd.Series(SORTEIO.permutation(valores)),
            lambda p: promessa.entrega(p, JANELA_ENTREGA)["taxa"], PEDACO_ESCADA),
            None, None)["dispersao"]))
        ream["ar"].append(float(estabilidade.resumo(estabilidade.por_janela(
            pd.Series(lei.simular(np.full(valores.size, a_global), sigma, SORTEIO)),
            lambda p: promessa.entrega(p, JANELA_ENTREGA)["taxa"], PEDACO_ESCADA),
            None, None)["dispersao"]))
        ream["tvar"].append(float(estabilidade.resumo(estabilidade.por_janela(
            pd.Series(lei.simular(MERCADOS[arquivo]["curvas"][252], escalas[252], SORTEIO)),
            lambda p: promessa.entrega(p, JANELA_ENTREGA)["taxa"], PEDACO_ESCADA),
            None, None)["dispersao"]))
    for nome, lista in ream.items():
        degraus[nome] = {"mediana": float(np.median(lista)),
                         "piso": float(np.percentile(lista, 10)),
                         "teto": float(np.percentile(lista, 90))}
    ESCADA[arquivo] = degraus
    rotulo = ROTULOS_SERIES[arquivo]
    print("%-11s | medida %.4f | independente %.4f | ar %.4f | tvar %.4f"
          % (rotulo, degraus["medida"], degraus["independente"]["mediana"],
             degraus["ar"]["mediana"], degraus["tvar"]["mediana"]))

CRITERIO_ESCADA = all(ESCADA[a]["tvar"]["mediana"] > ESCADA[a]["ar"]["mediana"] > ESCADA[a]["independente"]["mediana"]
                      for a in SERIES)
print("criterio 2 (a escada sobe degrau a degrau):", "SEPARA" if CRITERIO_ESCADA else "NAO SEPARA")

sp          | medida 0.0323 | independente 0.0127 | ar 0.0116 | tvar 0.0283


ibov        | medida 0.0245 | independente 0.0118 | ar 0.0103 | tvar 0.0174


btc         | medida 0.0242 | independente 0.0107 | ar 0.0114 | tvar 0.0133
criterio 2 (a escada sobe degrau a degrau): NAO SEPARA


In [6]:
# A região: quantos membros tvAR reproduzem as quatro estatísticas da geometria.
REGIAO = {}
for arquivo in MERCADOS:
    valores = volatilidade.retornos_log(dados.carregar_serie(arquivo)).dropna().to_numpy()
    sigma = MERCADOS[arquivo]["sigma"]
    real = regimes.estatisticas(valores, evidencia.JANELA_PADRAO, evidencia.POSTO_PADRAO, evidencia.BLOCO_PADRAO)
    membros = []
    for janela in JANELAS_TVAR:
        for cauda in (None, 3.0):
            folgas = []
            for _ in range(MUNDOS_REGIAO):
                mundo = lei.simular(MERCADOS[arquivo]["curvas"][janela],
                                    MERCADOS[arquivo]["escalas"][janela], SORTEIO, cauda=cauda)
                e = regimes.estatisticas(mundo, evidencia.JANELA_PADRAO, evidencia.POSTO_PADRAO, evidencia.BLOCO_PADRAO)
                d = {k: abs(e[k] - real[k]) / evidencia.TOLERANCIA_PADRAO[k] for k in evidencia.CHAVES_PADRAO}
                folgas.append(float(max(d.values())))
            membros.append({"janela": janela, "cauda": cauda,
                            "folga_mediana": float(np.median(folgas)),
                            "reproduz": bool(np.median(folgas) <= 1.0)})
    REGIAO[arquivo] = {"real": real, "membros": membros}
    cabem = sum(1 for m in membros if m["reproduz"])
    print("%-11s | membros tvAR que reproduzem as quatro: %d de %d | folgas %s"
          % (ROTULOS_SERIES[arquivo], cabem, len(membros),
             [("t" if m["cauda"] else "n") + " %.2f" % m["folga_mediana"] for m in membros]))

sp          | membros tvAR que reproduzem as quatro: 0 de 8 | folgas ['n 1.47', 't 1.52', 'n 2.47', 't 2.50', 'n 3.00', 't 4.00', 'n 3.00', 't 4.00']


ibov        | membros tvAR que reproduzem as quatro: 0 de 8 | folgas ['n 1.04', 't 1.69', 'n 1.50', 't 2.25', 'n 3.00', 't 3.75', 'n 4.00', 't 4.00']


btc         | membros tvAR que reproduzem as quatro: 0 de 8 | folgas ['n 1.50', 't 1.50', 'n 1.02', 't 2.00', 'n 2.00', 't 2.25', 'n 2.50', 't 2.85']


## As figuras

In [7]:
# Figura 1: a varredura --- o erro e a fração contra a janela, forma por forma.
CORES_FORMA = {"parado": "#7f7f7f", "degrau": "#1f4e79", "rampa": "#2e7d32", "andando": "#b03a2e"}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.4, 4.2))
for forma in FORMAS:
    js = [l["janela"] for l in VARREDURA[forma]["janelas"]]
    ax1.plot(js, [l["erro"] for l in VARREDURA[forma]["janelas"]],
             color=CORES_FORMA[forma], lw=1.6, label=forma)
    ax2.plot(js, [l["fracao"] for l in VARREDURA[forma]["janelas"]],
             color=CORES_FORMA[forma], lw=1.6)
for ax, titulo in ((ax1, "erro da escala (mediana)"), (ax2, "fração na tolerância (mediana)")):
    ax.set_xscale("log")
    ax.set_xlabel("janela do estimador (dias)")
    ax.set_title(titulo, fontsize=10)
    ax.grid(alpha=0.25)
ax1.legend(frameon=False, fontsize=8)
fig.tight_layout()
graficos.salvar(fig, "E34_janela", 1)
plt.close(fig)

In [8]:
# Figura 2: a escada --- a dispersão entre pedaços, degrau a degrau, contra a medida.
fig, eixos = plt.subplots(1, 3, figsize=(9.4, 3.6), sharey=True)
NOMES_ESCADA = [("independente", "dias embaralhados"), ("ar", "AR fixo"), ("tvar", "tvAR ajustado")]
for ax, arquivo in zip(eixos, SERIES):
    degraus = ESCADA[arquivo]
    altura = [degraus[n]["mediana"] for n, _ in NOMES_ESCADA]
    pisos = [degraus[n]["piso"] for n, _ in NOMES_ESCADA]
    tetos = [degraus[n]["teto"] for n, _ in NOMES_ESCADA]
    ax.bar([0, 1, 2], altura, 0.62, color=["#7f7f7f", "#1f4e79", "#b03a2e"])
    ax.errorbar([0, 1, 2], altura, yerr=[np.array(altura) - np.array(pisos),
                                         np.array(tetos) - np.array(altura)],
                fmt="none", ecolor="0.2", lw=1.2, capsize=3)
    ax.axhline(degraus["medida"], color="0.1", ls="--", lw=1.4, label="a medida")
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels([r for _, r in NOMES_ESCADA], fontsize=8)
    ax.set_title(ROTULOS_SERIES[arquivo], fontsize=10)
    ax.grid(alpha=0.25, axis="y")
eixos[0].set_ylabel("dispersão entre pedaços de dois anos")
eixos[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
graficos.salvar(fig, "E34_janela", 2)
plt.close(fig)

In [9]:
# Figura 3: a lei que anda --- a escala de cada dia nos três mercados, contra o chão independente.
razoes_chao = []
for _ in range(REAMOSTRAS):
    calmo = SORTEIO.normal(0.0, 1.0, 2000)
    escalas_calmo = volatilidade.volatilidade_rolante(pd.Series(calmo), 252, dias_uteis=1).to_numpy()
    razoes_chao.append(escalas_calmo / np.median(escalas_calmo[np.isfinite(escalas_calmo)]))
chao = np.concatenate([r[np.isfinite(r)] for r in razoes_chao])
piso_chao, teto_chao = np.percentile(chao, [2.5, 97.5])
fig, eixos = plt.subplots(1, 3, figsize=(9.4, 3.4), sharey=True)
for ax, arquivo in zip(eixos, SERIES):
    curva = MERCADOS[arquivo]["escalas"][252]
    razao = curva / np.median(curva[np.isfinite(curva)])
    anos = np.arange(curva.size) / 252.0
    ax.plot(anos, razao, color="#1f4e79", lw=0.9)
    ax.axhline(1.0, color="0.6", lw=0.8)
    ax.axhspan(piso_chao, teto_chao, color="0.6", alpha=0.18,
               label="o chão de mundos independentes")
    ax.set_title(ROTULOS_SERIES[arquivo], fontsize=10)
    ax.set_xlabel("anos")
    ax.grid(alpha=0.25)
eixos[0].set_ylabel("escala da janela de 252 dias, razão à mediana")
eixos[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
graficos.salvar(fig, "E34_janela", 3)
plt.close(fig)

## Leitura visual das figuras

(Feita contra o PNG de cada figura, com o código ao lado.)

**Figura 1** --- À esquerda, o erro das formas que mudam desce em U, com o fundo do vale em 504 dias
e a curva subindo de novo em 2016; a do mundo parado desce sem virar até o fim do eixo. À direita, a
fração na tolerância cai para perto de 0,65 na janela de 21 dias --- o desvio de janela curta é
barulhento demais para a tolerância declarada --- e chega a 1,0 a partir de 252 em todas as formas:
a tolerância não distingue as formas, quem distingue é o erro.

**Figura 2** --- No sp, a barra vermelha do tvAR é a mais alta e quase toca a linha da medida; no
ibov, ela fica no meio do caminho; no btc, as três barras ficam juntas e todas longe da linha.

**Figura 3** --- A escala de cada dia caminha: no sp e no ibov chega perto de três vezes a mediana,
no btc perto de duas e meia --- sempre muito além da banda cinza do chão de mundos independentes,
que é estreita em torno de um.

In [10]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
resultado = {
    "janela_semente": SEMENTE,
    "janela_mundos": MUNDOS_VARREDURA,
    "janela_tolerancia": TOLERANCIA,
    "janela_n_mundo": N_MUNDO,
    "janela_quando": QUANDO_MUNDO,
    "janela_fator": FATOR_MUNDO,
    "janela_dias_rampa": DIAS_RAMPA_CURTA,
    "janela_reamostras": REAMOSTRAS,
    "janela_mundos_regiao": MUNDOS_REGIAO,
    "janela_criterio_formas": int(CRITERIO_FORMAS),
    "janela_criterio_escada": int(CRITERIO_ESCADA),
}
for forma in FORMAS:
    resultado["janela_%s_erro_primeiro_ano" % forma] = round(ABERTURA[forma]["erro_primeiro_ano"], 4)
    resultado["janela_%s_erro_ultimo_trecho" % forma] = round(ABERTURA[forma]["erro_ultimo_trecho"], 4)
    resultado["janela_%s_fracao" % forma] = round(ABERTURA[forma]["fracao"], 3)
    resultado["janela_%s_otimo_mediana" % forma] = OTIMOS[forma]["mediana"]
    resultado["janela_%s_otimo_piso" % forma] = OTIMOS[forma]["piso"]
    resultado["janela_%s_otimo_teto" % forma] = OTIMOS[forma]["teto"]
    melhor_taxa = min(VARREDURA[forma]["taxas"], key=lambda l: l["erro"])
    resultado["janela_%s_memoria_vencedora" % forma] = round(1.0 / melhor_taxa["taxa"], 1)
    resultado["janela_%s_erro_vencedor" % forma] = round(melhor_taxa["erro"], 4)
    for l in VARREDURA[forma]["janelas"]:
        chave = {21: "vinte_um", 63: "sessenta_tres", 126: "cento_vinte_seis", 252: "duzentos_cinquenta_dois",
                 504: "quinhentos_quatro", 1008: "mil_oito", 2016: "dois_mil_dezesseis"}.get(l["janela"])
        if chave and l["janela"] in (21, 252, 2016):
            resultado["janela_%s_erro_%s" % (forma, chave)] = round(l["erro"], 4)
for arquivo in SERIES:
    rotulo = ROTULOS_SERIES[arquivo]
    m = MERCADOS[arquivo]
    resultado["janela_%s_dias" % rotulo] = m["dias"]
    resultado["janela_%s_a_global" % rotulo] = round(m["a_global"], 3)
    resultado["janela_%s_orcamento" % rotulo] = round(m["orcamento_escala"], 2)
    resultado["janela_%s_orcamento_chao" % rotulo] = round(m["orcamento_escala_chao"], 2)
    d = ESCADA[arquivo]
    resultado["janela_escada_%s_medida" % rotulo] = round(d["medida"], 4)
    for nome in ("independente", "ar", "tvar"):
        resultado["janela_escada_%s_%s" % (rotulo, nome)] = round(d[nome]["mediana"], 4)
    resultado["janela_regiao_%s_cabem" % rotulo] = sum(1 for mm in REGIAO[arquivo]["membros"] if mm["reproduz"])
    resultado["janela_regiao_%s_tentados" % rotulo] = len(REGIAO[arquivo]["membros"])
    for mm in REGIAO[arquivo]["membros"]:
        chave = {63: "sessenta_tres", 126: "cento_vinte_seis", 252: "duzentos_cinquenta_dois",
                 504: "quinhentos_quatro"}[mm["janela"]]
        sufixo = "t" if mm["cauda"] is not None else "normal"
        resultado["janela_regiao_%s_folga_%s_%s" % (rotulo, chave, sufixo)] = round(mm["folga_mediana"], 2)

# O que sai do laboratório e o que o livro cita: medida que o livro não usa é medida morta.
CITADAS_NO_LIVRO = (
    "janela_andando_erro_dois_mil_dezesseis", "janela_andando_erro_duzentos_cinquenta_dois",
    "janela_andando_erro_vinte_um", "janela_andando_memoria_vencedora",
    "janela_andando_otimo_mediana", "janela_btc_a_global", "janela_btc_orcamento",
    "janela_btc_orcamento_chao", "janela_degrau_otimo_mediana", "janela_escada_btc_ar",
    "janela_escada_btc_independente", "janela_escada_btc_medida", "janela_escada_btc_tvar",
    "janela_escada_ibov_ar", "janela_escada_ibov_independente", "janela_escada_ibov_medida",
    "janela_escada_ibov_tvar", "janela_escada_sp_ar", "janela_escada_sp_independente",
    "janela_escada_sp_medida", "janela_escada_sp_tvar", "janela_ibov_a_global",
    "janela_ibov_orcamento", "janela_ibov_orcamento_chao", "janela_mundos_regiao",
    "janela_parado_erro_dois_mil_dezesseis", "janela_parado_erro_duzentos_cinquenta_dois",
    "janela_parado_erro_vinte_um", "janela_parado_fracao", "janela_parado_memoria_vencedora",
    "janela_parado_otimo_mediana", "janela_rampa_otimo_mediana", "janela_reamostras",
    "janela_regiao_btc_folga_sessenta_tres_normal", "janela_regiao_ibov_folga_sessenta_tres_normal",
    "janela_regiao_sp_folga_sessenta_tres_normal", "janela_regiao_sp_tentados",
    "janela_sp_a_global", "janela_sp_orcamento", "janela_sp_orcamento_chao",
)
resultado = {chave: valor for chave, valor in resultado.items() if chave in CITADAS_NO_LIVRO}

caminho = Path("lab/resultados/E34_janela.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E34_janela.json gravado | 40 grandezas
